# 03 — Meta-learner Comparison: S / T / X / R / DR

**Narrative thread:** Treatment imbalance (~85% treated) is the design challenge.
Each learner's failure mode is characterized relative to that imbalance.

**Why X-learner should win here:**
- T-learner's mu0 is noisy (15% of data)
- X-learner imputes TEs from the large arm: D0 = mu1(X_control) - Y_control
- Propensity weighting puts 85% weight on tau0 (estimated via D0, the *large* arm data)
- This is the variance correction T-learner lacks

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.data import load_data, split, get_Xyt
from src.learners import LEARNERS
from src.metrics import evaluate_all, qini_curve, qini_coefficient, uplift_by_decile
import time

plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})

In [ ]:
df = load_data(percent10=True)
train, val, test = split(df)
X_tr, y_tr, t_tr = get_Xyt(train)
X_te, y_te, t_te = get_Xyt(test)
print(f'Train: {len(train):,}  |  Test: {len(test):,}')
print(f'Treatment ratio — train: {t_tr.mean():.3f}, test: {t_te.mean():.3f}')

## Train all meta-learners

In [ ]:
results = []
predictions = {}

for name, LearnerClass in LEARNERS.items():
    print(f'Training {name}...', end=' ')
    t0 = time.time()
    model = LearnerClass()
    model.fit(X_tr, y_tr, t_tr)
    tau_hat = model.predict(X_te)
    predictions[name] = tau_hat
    metrics = evaluate_all(y_te, tau_hat, t_te, name)
    metrics['train_time_s'] = round(time.time() - t0, 1)
    results.append(metrics)
    print(f'done ({metrics["train_time_s"]}s)  Qini={metrics["qini_coeff"]:.4f}')

## Comparison table

In [ ]:
res_df = pd.DataFrame(results).set_index('model')
res_df = res_df.sort_values('qini_coeff', ascending=False)

# Bold the winner
print('=== Meta-learner Comparison (test set) ===')
print(res_df[['qini_coeff', 'uplift@10', 'uplift@20', 'uplift@30', 'auuc']].round(5).to_string())
print(f'\nBest model: {res_df["qini_coeff"].idxmax()} (Qini={res_df["qini_coeff"].max():.4f})')

## Qini curves — all learners on one plot

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
colors = plt.cm.tab10(np.linspace(0, 1, len(predictions)))

for (name, tau_hat), color in zip(predictions.items(), colors):
    x, q = qini_curve(y_te, tau_hat, t_te)
    frac = x / len(y_te)
    qc = qini_coefficient(y_te, tau_hat, t_te)
    ax.plot(frac, q, label=f'{name}  ({qc:.3f})', color=color, linewidth=1.8)

# random
q_last = qini_curve(y_te, list(predictions.values())[0], t_te)[1][-1]
ax.plot([0, 1], [0, q_last], 'k--', linewidth=1.0, label='Random')

ax.set_xlabel('Fraction of population targeted')
ax.set_ylabel('Qini score')
ax.set_title('Qini curves — all meta-learners  (Qini coeff in legend)')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/qini_comparison.png', bbox_inches='tight')
plt.show()

## Uplift by decile — X-learner (headline model)

In [ ]:
best_name = res_df['qini_coeff'].idxmax()
best_tau = predictions[best_name]

bins, realized = uplift_by_decile(y_te, best_tau, t_te)

fig, ax = plt.subplots(figsize=(8, 4))
colors_d = ['#d62728' if v < 0 else '#1f77b4' for v in realized]
ax.bar(bins, realized * 100, color=colors_d)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Decile (1 = highest predicted uplift)')
ax.set_ylabel('Realized uplift (%)')
ax.set_title(f'Uplift by decile — {best_name}\n(monotone decrease = model genuinely ranks persuadables)')
ax.set_xticks(bins)
plt.tight_layout()
plt.savefig('../reports/uplift_by_decile.png', bbox_inches='tight')
plt.show()

## Failure mode analysis

In [ ]:
# S-learner: does it shrink uplift estimates toward 0?
s_tau = predictions.get('S-learner', None)
x_tau = predictions.get('X-learner', None)
t_tau = predictions.get('T-learner', None)

print('Uplift estimate variance comparison (proxy for S-learner shrinkage):')
for name, tau in predictions.items():
    print(f'  {name:12s}: std={np.std(tau):.5f},  mean={np.mean(tau):.5f}')
print()
print('S-learner should show smallest std — regularization shrinks T coefficient toward 0.')
print('This is especially problematic here: base rate 4.7%, small absolute lift => T is a weak signal.')

In [ ]:
# Save results for app
import pickle, pathlib
pathlib.Path('../data').mkdir(exist_ok=True)
with open('../data/meta_learner_predictions.pkl', 'wb') as f:
    pickle.dump({
        'predictions': {k: v for k, v in predictions.items()},
        'y_te': y_te,
        't_te': t_te,
        'results': results
    }, f)
print('Saved predictions to data/meta_learner_predictions.pkl')